# Phase 5 — HITL Review, Fraud Memory & Audit Trail

**Project:** Fraud AI Investigator — MENA Fintech Portfolio  
**Notebook:** `notebooks/hitl_review.ipynb`  
**Author:** Ahmed Raza  
**Last updated:** 2026-05

---

## Objective

Validate and demonstrate the Phase 5 HITL review pipeline:

1. **Is the analyst review queue working?** — `GET /v1/hitl/queue`
2. **Does the context endpoint return everything needed?** — investigation summary, past cases, audit trail
3. **Does a HITL decision correctly update alert status and memory?** — end-to-end verdict submission
4. **Does fraud memory improve synthesis context?** — show past cases appearing in investigation
5. **Is the audit trail complete?** — every event from creation to analyst decision

## Complete pipeline

```
POST /v1/alerts/generate    ← rules fire, alerts created (PENDING)
POST /v1/triage/batch       ← LLM scores each alert
POST /v1/investigate/batch  ← 5-agent LangGraph investigation
GET  /v1/hitl/queue         ← analyst sees AWAITING_HUMAN alerts
GET  /v1/hitl/{id}/context  ← analyst reads full context + past cases
POST /v1/hitl/{id}/decision ← analyst submits verdict
GET  /v1/alerts/{id}/audit  ← complete immutable evidence trail
```

## Prerequisites

```bash
uv run uvicorn app.main:app --reload
uv run jupyter notebook notebooks/hitl_review.ipynb
```

In [ ]:
import sys
import warnings
from datetime import datetime
from pathlib import Path

import requests
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
warnings.filterwarnings('ignore', category=DeprecationWarning)

plt.style.use('seaborn-v0_8-whitegrid')

BASE_URL        = 'http://localhost:8000'
SCREENSHOTS_DIR = PROJECT_ROOT / 'doc' / 'Screenshots'
SCREENSHOTS_DIR.mkdir(parents=True, exist_ok=True)

try:
    health = requests.get(f'{BASE_URL}/health', timeout=5).json()
    assert health['status'] == 'ok'
    print('API ready ✓')
except Exception:
    raise RuntimeError('API not running. Start: uv run uvicorn app.main:app --reload')

---
## Section 1 — Run the full pipeline to generate an AWAITING_HUMAN alert

In [ ]:
import time

print('Step 1: Generate alerts...')
gen = requests.post(f'{BASE_URL}/v1/alerts/generate', json={'limit': 10}).json()
print(f"  Created: {gen['alerts_created']} alerts")

print('\nStep 2: Triage alerts (LLM)...')
triage = requests.post(f'{BASE_URL}/v1/triage/batch', json={'max_alerts': 3}, timeout=120).json()
print(f"  Triaged: {triage['processed']} | Escalated: {triage['escalated']}")

print('\nStep 3: Run LangGraph investigation...')
inv = requests.post(f'{BASE_URL}/v1/investigate/batch', json={'max_alerts': 2}, timeout=120).json()
print(f"  Investigated: {inv['investigated']} | Succeeded: {inv['succeeded']}")

print('\nStep 4: Check HITL queue...')
queue = requests.get(f'{BASE_URL}/v1/hitl/queue').json()
print(f"  Queue length: {queue['queue_length']}")
for a in queue['alerts']:
    print(f"  Alert: {a['alert_id']} | score={a['risk_score']} | trigger={a['trigger']}")

---
## Section 2 — Review context for the highest-risk alert

In [ ]:
target_id = None

if queue['alerts']:
    target_id = queue['alerts'][0]['alert_id']
    context   = requests.get(f'{BASE_URL}/v1/hitl/{target_id}/context').json()

    print('ANALYST REVIEW CONTEXT')
    print('=' * 65)
    a = context['alert']
    print(f"Alert ID    : {a['alert_id']}")
    print(f"Risk Score  : {a['risk_score']}/100  ({a['risk_band']})")
    print(f"Trigger     : {a['trigger']}")
    print(f"Status      : {a['status']}")
    print()
    print('INVESTIGATION SUMMARY:')
    print(a.get('investigation_summary', 'Not yet investigated'))
    print()
    print(f"SIMILAR PAST CASES: {len(context['similar_past_cases'])}")
    for case in context['similar_past_cases']:
        print(f"  [{case['verdict']}] customer={case['customer_id']} score={case['risk_score']}")
    print()
    print('REGULATORY GUIDANCE:')
    for g in context['regulatory_guidance']:
        print(f"  ⚖ {g}")
    print()
    print(f"VALID VERDICTS: {context['valid_verdicts']}")
    print(f"AUDIT EVENTS  : {context['audit_event_count']}")
else:
    print('No alerts in HITL queue.')
    print('Tip: run triage with escalation threshold lowered, or generate more alerts.')

---
## Section 3 — Submit analyst verdict

In [ ]:
decision_result = None

if target_id:
    print(f'Submitting verdict for alert: {target_id}')
    r = requests.post(
        f'{BASE_URL}/v1/hitl/{target_id}/decision',
        json={
            'verdict' : 'CONFIRMED_FRAUD',
            'analyst' : 'senior.analyst@bank.ae',
            'notes'   : (
                'Multiple signals confirmed: high-value transfer to FATF-listed '
                'jurisdiction, device mismatch indicating account takeover, '
                'merchant name matches OFAC SDN entity. STR to be filed within 2 working days.'
            ),
            'risk_signals': ['High value AED 55k', 'Iran FATF corridor', 'OFAC SDN match'],
        },
    )
    decision_result = r.json()

    print('HITL DECISION RESULT')
    print('=' * 50)
    print(f"Verdict       : {decision_result['verdict']}")
    print(f"New status    : {decision_result['new_status']}")
    print(f"STR required  : {decision_result['str_required']}")
    print(f"STR deadline  : {decision_result.get('str_deadline')}")
    print(f"Memory ID     : {decision_result['memory_id']}")
    print(f"Analyst       : {decision_result['analyst']}")
    print(f"Processed at  : {decision_result['processed_at']}")
else:
    print('No target alert — run Section 1 first')

---
## Section 4 — Verify fraud memory and complete audit trail

In [ ]:
# Fraud memory stats
mem_stats = requests.get(f'{BASE_URL}/v1/hitl/memory/stats').json()
print('FRAUD MEMORY')
print('=' * 40)
print(f"Total cases       : {mem_stats['total_cases']}")
print(f"Confirmed fraud   : {mem_stats['confirmed_fraud']}")
print(f"False positives   : {mem_stats['false_positives']}")
print(f"Unique customers  : {mem_stats['unique_customers']}")

if target_id:
    print()
    print('COMPLETE AUDIT TRAIL')
    print('=' * 65)
    audit = requests.get(f'{BASE_URL}/v1/alerts/{target_id}/audit').json()
    print(f"Total events: {audit['event_count']}")
    for event in audit['events']:
        print(f"  [{event['timestamp'][:19]}] {event['event_type']:<30} actor={event['actor']}")

---
## Section 5 — Pipeline outcome visualisation

In [ ]:
stats = requests.get(f'{BASE_URL}/v1/investigate/stats').json()

status_counts = {
    'PENDING'         : stats.get('pending', 0),
    'AUTO_CLOSED'     : stats.get('auto_closed', 0),
    'INVESTIGATING'   : stats.get('investigating', 0),
    'AWAITING_HUMAN'  : stats.get('awaiting_human', 0),
    'FRAUD_CONFIRMED' : stats.get('fraud_confirmed', 0),
    'FALSE_POSITIVE'  : stats.get('false_positive', 0),
}
status_counts = {k: v for k, v in status_counts.items() if v > 0}

if status_counts:
    colors = {
        'PENDING'         : '#9E9E9E',
        'AUTO_CLOSED'     : '#4CAF50',
        'INVESTIGATING'   : '#FF9800',
        'AWAITING_HUMAN'  : '#1565C0',
        'FRAUD_CONFIRMED' : '#D32F2F',
        'FALSE_POSITIVE'  : '#8BC34A',
    }
    fig, ax = plt.subplots(figsize=(10, 4))
    bar_colors = [colors.get(k, '#888') for k in status_counts]
    bars = ax.bar(status_counts.keys(), status_counts.values(),
                  color=bar_colors, alpha=0.85, edgecolor='white')
    for bar, (k, v) in zip(bars, status_counts.items()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                str(v), ha='center', fontsize=11, fontweight='500')
    ax.set_title('Alert pipeline status — end-to-end (Phase 1 → Phase 5)', fontsize=12)
    ax.set_ylabel('Alert count')
    ax.set_xticklabels(status_counts.keys(), rotation=15, ha='right')
    plt.tight_layout()
    save_path = SCREENSHOTS_DIR / '10_pipeline_status.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Chart saved: {save_path}')
else:
    print('No alerts yet — run Section 1 first')

---
## Section 6 — Completion checklist

In [ ]:
print('PHASE 5 NOTEBOOK — COMPLETION CHECKLIST')
print('=' * 55)

checks = {
    'API health check passed'            : health['status'] == 'ok',
    'HITL queue endpoint works'          : requests.get(f'{BASE_URL}/v1/hitl/queue').status_code == 200,
    'Memory stats endpoint works'        : requests.get(f'{BASE_URL}/v1/hitl/memory/stats').status_code == 200,
    'Alerts generated and triaged'       : gen.get('alerts_created', 0) > 0,
    'Investigation ran'                  : inv.get('succeeded', 0) > 0,
    'HITL verdict submitted'             : decision_result is not None,
    'STR obligation flagged correctly'   : decision_result.get('str_required', False) if decision_result else False,
    'Memory recorded after verdict'      : mem_stats.get('total_cases', 0) > 0,
    'Pipeline status chart saved'        : (SCREENSHOTS_DIR / '10_pipeline_status.png').exists(),
}

all_passed = True
for label, passed in checks.items():
    print(f"  {'✓' if passed else '✗'}  {label}")
    if not passed:
        all_passed = False

print()
if all_passed:
    print('All checks passed — Phase 5 complete ✓')
    print('Ready for Phase 6: Streamlit dashboard + Docker packaging.')
else:
    print('Some checks failed — run Sections 1-4 in order.')

print(f'\nCompleted: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')